### Inspecting the columns

In [2]:
import pandas as pd

team_raw = pd.read_csv("/Users/leahkim/Desktop/project1-sports-outcomes/raw/team.csv")
game_raw = pd.read_csv("/Users/leahkim/Desktop/project1-sports-outcomes/raw/game.csv")
line_score_raw = pd.read_csv("/Users/leahkim/Desktop/project1-sports-outcomes/raw/line_score.csv")

print("team.csv columns:")
print(team_raw.columns.tolist())
print()

print("game.csv columns:")
print(game_raw.columns.tolist())
print()

print("line_score.csv columns:")
print(line_score_raw.columns.tolist())

team.csv columns:
['id', 'full_name', 'abbreviation', 'nickname', 'city', 'state', 'year_founded']

game.csv columns:
['season_id', 'team_id_home', 'team_abbreviation_home', 'team_name_home', 'game_id', 'game_date', 'matchup_home', 'wl_home', 'min', 'fgm_home', 'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home', 'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home', 'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home', 'pts_home', 'plus_minus_home', 'video_available_home', 'team_id_away', 'team_abbreviation_away', 'team_name_away', 'matchup_away', 'wl_away', 'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away', 'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away', 'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away', 'pf_away', 'pts_away', 'plus_minus_away', 'video_available_away', 'season_type']

line_score.csv columns:
['game_date_est', 'game_sequence', 'game_id', 'team_id_home', 'team_abbreviatio

### Create the Tables

In [ ]:
import pandas as pd
from pathlib import Path

# Paths
RAW_DIR = Path("raw")
OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)

# Load raw files
team_raw = pd.read_csv(RAW_DIR / "team.csv")
game_raw = pd.read_csv(RAW_DIR / "game.csv")
line_score_raw = pd.read_csv(RAW_DIR / "line_score.csv")

print("Loaded raw files:")
print("team.csv:", team_raw.shape)
print("game.csv:", game_raw.shape)
print("line_score.csv:", line_score_raw.shape)

# -------------------------
# 1. Create teams table
# -------------------------
teams = team_raw.rename(columns={
    "id": "team_id",
    "full_name": "team_name"
})[
    ["team_id", "team_name", "abbreviation", "nickname", "city", "state", "year_founded"]
].drop_duplicates()

# -------------------------
# 2. Create seasons table
# -------------------------
seasons = game_raw[["season_id", "season_type"]].drop_duplicates().copy()
seasons["season_year"] = seasons["season_id"]

# Reorder columns
seasons = seasons[["season_id", "season_year", "season_type"]]

# -------------------------
# 3. Create games table
# -------------------------
games = game_raw.rename(columns={
    "team_id_home": "home_team_id",
    "team_id_away": "away_team_id",
    "pts_home": "home_score",
    "pts_away": "away_score"
})[
    ["game_id", "game_date", "season_id", "season_type", "home_team_id", "away_team_id", "home_score", "away_score"]
].drop_duplicates(subset=["game_id"]).copy()

games["winner_team_id"] = games.apply(
    lambda row: row["home_team_id"] if row["home_score"] > row["away_score"] else row["away_team_id"],
    axis=1
)

# -------------------------
# 4. Create team_game_stats table
# -------------------------
home_stats = game_raw[[
    "game_id",
    "team_id_home",
    "fg_pct_home",
    "ft_pct_home",
    "fg3_pct_home",
    "ast_home",
    "reb_home",
    "tov_home",
    "pts_home",
    "wl_home"
]].copy()

home_stats = home_stats.rename(columns={
    "team_id_home": "team_id",
    "fg_pct_home": "fg_pct",
    "ft_pct_home": "ft_pct",
    "fg3_pct_home": "fg3_pct",
    "ast_home": "ast",
    "reb_home": "reb",
    "tov_home": "tov",
    "pts_home": "pts",
    "wl_home": "wl"
})

away_stats = game_raw[[
    "game_id",
    "team_id_away",
    "fg_pct_away",
    "ft_pct_away",
    "fg3_pct_away",
    "ast_away",
    "reb_away",
    "tov_away",
    "pts_away",
    "wl_away"
]].copy()

away_stats = away_stats.rename(columns={
    "team_id_away": "team_id",
    "fg_pct_away": "fg_pct",
    "ft_pct_away": "ft_pct",
    "fg3_pct_away": "fg3_pct",
    "ast_away": "ast",
    "reb_away": "reb",
    "tov_away": "tov",
    "pts_away": "pts",
    "wl_away": "wl"
})

team_game_stats = pd.concat([home_stats, away_stats], ignore_index=True)

# Convert W/L to binary flag
team_game_stats["win_flag"] = team_game_stats["wl"].map({"W": 1, "L": 0})

# Add surrogate primary key
team_game_stats.insert(0, "team_game_stats_id", range(1, len(team_game_stats) + 1))

# Keep final columns
team_game_stats = team_game_stats[
    ["team_game_stats_id", "game_id", "team_id", "fg_pct", "ft_pct", "fg3_pct", "ast", "reb", "tov", "pts", "win_flag"]
]

# -------------------------
# Save outputs
# -------------------------
teams.to_csv(OUT_DIR / "teams.csv", index=False)
seasons.to_csv(OUT_DIR / "seasons.csv", index=False)
games.to_csv(OUT_DIR / "games.csv", index=False)
team_game_stats.to_csv(OUT_DIR / "team_game_stats.csv", index=False)

print("\nCreated tables:")
print("teams:", teams.shape)
print("seasons:", seasons.shape)
print("games:", games.shape)
print("team_game_stats:", team_game_stats.shape)